# FM回归
将AED与fm_reg_controls表进合并，用于进行回归  

## 导入库

In [14]:
import polars as pl
import dotenv
import os
import numpy as np
import pandas as pd
from statsmodels.regression.linear_model import OLS
import statsmodels.api as sm
from statsmodels.stats.sandwich_covariance import cov_hac
from statsmodels.tools.tools import add_constant
import warnings
dotenv.load_dotenv()

True

## 超参数

In [15]:
TASK_PREFIX = 'baseline1'
SAVE_BASE_DIR = f'/home/frank/files/programs/GraduationThesis/empirical/{TASK_PREFIX}'
SAVE = True
HETER = True # 如果不使用异质性，则导入MA_copy，保证不存在异质性

DATABASE_URL = os.getenv('POSTGRES_URL')

## 读取数据
- 从数据库中读取控制变量  
- 从BASE_LINE_REG_DIR中获取aed.parquet 

### 读数据库 

In [16]:
# 使用polars从数据库读取fm_reg_controls表数据，指定schema避免类型推断问题
fm_reg_controls_schema = {
    'stkcd': pl.Utf8,              # 股票代码
    'accper': pl.Date,             # 会计期间
    'betavals': pl.Float64,        # 贝塔值
    'bm_ratio': pl.Float64,        # 账面市值比
    'gross_margin': pl.Float64,    # 毛利率
    'investment_ratio': pl.Float64, # 投资比率
    'ln_market_value': pl.Float64   # 对数市值
}

fm_reg_controls_df = pl.read_database_uri(
    query="SELECT * FROM statics.fm_reg_controls",
    uri=DATABASE_URL,
    schema_overrides=fm_reg_controls_schema
)

fm_reg_controls_df.head()

stkcd,accper,betavals,bm_ratio,gross_margin,investment_ratio,ln_market_value
str,date,f64,f64,f64,f64,f64
"""000001""",1997-01-01,0.93259,null,null,null,16.435539
"""000001""",1997-02-01,0.88586,null,null,null,16.433457
"""000001""",1997-03-01,1.06864,null,null,null,16.785434
"""000001""",1997-04-01,1.01343,null,null,null,17.082685
"""000001""",1997-05-01,1.22579,null,null,null,16.994263


### 读取AED因子

In [17]:
ma_df = pl.read_parquet(SAVE_BASE_DIR + '/MA因子.parquet') if HETER else pl.read_parquet(SAVE_BASE_DIR + '/MA因子_copy.parquet')
ma_df.head()

date,portfolio,MA,return,finance_and_short
date,str,f64,f64,i32
2023-01-01,"""301019""",0.673006,0.0132,0
2017-03-01,"""600770""",0.607574,-0.1574,0
2024-03-01,"""300512""",0.839616,-0.0327,0
2018-12-01,"""300215""",0.748944,0.0412,0
2021-07-01,"""300092""",0.5874,0.234804,0


## 处理数据 
对于Controls数据，由于部分数据是季频，用该季数据填充该月数据  

In [18]:
### 季度数据填充

# 定义季度数据字段
quarterly_columns = ['bm_ratio', 'gross_margin', 'investment_ratio']

def expand_quarterly_to_monthly(df, quarterly_cols, date_col='accper', stkcd_col='stkcd'):
    """
    将季度数据扩展到月度，使用季度末数据填充该季度其他月份
    使用Polars原生操作避免Object类型问题
    """
    # 确保日期列是datetime类型并提取月份
    df = df.with_columns([
        pl.col(date_col).cast(pl.Date),
        pl.col(date_col).dt.month().alias('month')
    ])

    # 过滤出季度末数据
    quarter_end_data = df.filter(pl.col('month').is_in([3, 6, 9, 12]))

    # 为每个季度创建扩展数据
    expanded_dfs = []

    for quarter_end_month in [3, 6, 9, 12]:
        qtr_data = quarter_end_data.filter(pl.col('month') == quarter_end_month)

        # 确定该季度包含的月份
        if quarter_end_month == 3:      # Q1: 1,2,3月
            months = [1, 2, 3]
        elif quarter_end_month == 6:    # Q2: 4,5,6月
            months = [4, 5, 6]
        elif quarter_end_month == 9:    # Q3: 7,8,9月
            months = [7, 8, 9]
        else:  # 12 - Q4: 10,11,12月
            months = [10, 11, 12]

        # 为每个月创建数据
        for month in months:
            month_data = qtr_data.with_columns(
                pl.date(pl.col(date_col).dt.year(), pl.lit(month), pl.lit(1)).alias(date_col)
            )
            expanded_dfs.append(month_data)

    # 合并所有扩展数据
    if expanded_dfs:
        expanded_df = pl.concat(expanded_dfs)
        # 按股票代码和日期排序
        expanded_df = expanded_df.sort([stkcd_col, date_col])
        # 移除辅助列
        expanded_df = expanded_df.drop('month')
        return expanded_df
    else:
        return pl.DataFrame()

# 应用季度数据填充
fm_reg_controls_monthly_df = expand_quarterly_to_monthly(
    fm_reg_controls_df,
    quarterly_columns
)

# 重命名列
fm_reg_controls_monthly_df = fm_reg_controls_monthly_df.rename({'accper': 'date','stkcd':'portfolio'})

# 注意：后续分析请使用填充后的数据 fm_reg_controls_monthly_df
# 它包含了扩展到月度的季度财务数据
fm_reg_controls_monthly_df.head(10)

portfolio,date,betavals,bm_ratio,gross_margin,investment_ratio,ln_market_value
str,date,f64,f64,f64,f64,f64
"""000001""",1997-01-01,1.06864,null,null,null,16.785434
"""000001""",1997-02-01,1.06864,null,null,null,16.785434
"""000001""",1997-03-01,1.06864,null,null,null,16.785434
"""000001""",1997-04-01,1.06346,0.550942,null,0.978269,17.109268
"""000001""",1997-05-01,1.06346,0.550942,null,0.978269,17.109268
"""000001""",1997-06-01,1.06346,0.550942,null,0.978269,17.109268
"""000001""",1997-07-01,0.85352,null,null,null,16.943189
"""000001""",1997-08-01,0.85352,null,null,null,16.943189
"""000001""",1997-09-01,0.85352,null,null,null,16.943189


连接数据

In [19]:
ma_df = ma_df.join(fm_reg_controls_monthly_df, on=['date', 'portfolio'], how='left')
ma_df.head()

date,portfolio,MA,return,finance_and_short,betavals,bm_ratio,gross_margin,investment_ratio,ln_market_value
date,str,f64,f64,i32,f64,f64,f64,f64,f64
2023-01-01,"""301019""",0.673006,0.0132,0,0.67042,0.63391,0.3392,0.672657,14.202225
2023-01-01,"""301019""",0.673006,0.0132,0,0.67041,0.63391,0.3392,0.672657,14.202225
2017-03-01,"""600770""",0.607574,-0.1574,0,0.80934,0.496071,0.2965,0.697356,16.142855
2024-03-01,"""300512""",0.839616,-0.0327,0,1.31944,0.753975,0.3257,0.971126,14.595416
2024-03-01,"""300512""",0.839616,-0.0327,0,1.31015,0.753975,0.3257,0.971126,14.595416


## 回归

定义一个FM回归的函数

In [20]:
def fm_reg(
    df:pl.DataFrame, 
    return_col:str, 
    x_cols:list[str],
    date_col:str='date',
    stkcd_col:str='portfolio'
) -> pl.DataFrame:
    """
    进行FM回归

    FM回归：
        - 1. 对于每一个时间的截面数据，进行线性回归
        - 2. 对于所有回归系数序列，求均值，HAC-t，p

    参数：
        df: 包含时间序列和截面数据的DataFrame
        return_col: 被解释变量列名
        x_cols: 解释变量列名列表
        date_col: 日期列名，默认为'date'
        stkcd_col: 股票代码列名，默认为'portfolio'（目前未使用，预留扩展）
    """
    # 确保返回列和自变量列存在
    if return_col not in df.columns:
        raise ValueError(f"返回列 '{return_col}' 不存在")

    for col in x_cols:
        if col not in df.columns:
            raise ValueError(f"自变量列 '{col}' 不存在")

    df = df.__copy__()
    df = df.drop_nulls()

    # 获取所有时间点
    time_points = df.select(date_col).unique().sort(date_col).to_series().to_list()

    # 存储每个时间点的回归系数
    coefficients_data = []

    for time_point in time_points:
        # 获取该时间点的截面数据
        cross_section = df.filter(pl.col(date_col) == time_point)

        # 跳过数据不足的时间点
        if len(cross_section) < len(x_cols) + 1:  # 需要至少比变量数多1个观测值
            warnings.warn(f"时间点 {time_point} 数据不足，跳过回归")
            continue

        # 准备回归数据
        y = cross_section.select(return_col).to_pandas().values.flatten()
        X = cross_section.select(x_cols).to_pandas().values

        # 添加常数项
        X = sm.add_constant(X)

        # 检查是否有足够的有效观测值
        valid_mask = ~np.isnan(y)
        for i in range(X.shape[1]):
            valid_mask = valid_mask & ~np.isnan(X[:, i])

        if np.sum(valid_mask) < len(x_cols) + 1:
            continue

        y_clean = y[valid_mask]
        X_clean = X[valid_mask]

        try:
            # 进行OLS回归
            model = OLS(y_clean, X_clean)
            results = model.fit()

            # 存储回归系数
            coeff_dict = {'date': time_point, 'const': results.params[0], 'n_obs': len(y_clean)}

            # 添加自变量系数
            for i, col in enumerate(x_cols):
                coeff_dict[col] = results.params[i + 1]

            coefficients_data.append(coeff_dict)

        except Exception as e:
            # 如果回归失败，跳过该时间点
            continue

    if not coefficients_data:
        raise ValueError("没有足够的有效数据进行回归")

    # 转换为DataFrame
    coeff_df = pd.DataFrame(coefficients_data)

    # 计算统计量
    stats_results = []

    # 对每个系数（包括常数项和自变量）计算统计量
    coeff_cols = ['const'] + x_cols

    for col in coeff_cols:
        if col not in coeff_df.columns:
            continue

        # 获取系数时间序列
        series = coeff_df[col].dropna()

        if len(series) < 2:
            continue

        # 计算HAC标准误差和t统计量
        try:
            y = series.to_numpy()
            x = np.ones((len(y), 1))
            results = sm.OLS(y, x).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
            mean_coeff = results.params[0]
            t_stat = results.tvalues[0]
            p_value = results.pvalues[0]

        except Exception:
            warnings.warn(f"时间点 {time_point} 回归失败，跳过")
            t_stat = np.nan
            p_value = np.nan

        stats_results.append({
            'variable': col,
            'mean': mean_coeff,
            't_stat': t_stat,
            'p_value': p_value,
        })

        n_periods = len(series)

    # 转换为Polars DataFrame
    result_df = pl.DataFrame(stats_results)
    result_df = result_df.with_columns(
        pl.col('mean').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('mean'),
        pl.col('t_stat').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('t_stat'),
        pl.col('p_value').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('p_value'),
    )

    # t、p 加括号
    result_df = result_df.select(
        pl.col('variable'),
        pl.col('mean'),
        (pl.lit('[') + pl.col('t_stat') + pl.lit(']')).alias('t_stat'),
        (pl.lit('(') + pl.col('p_value') + pl.lit(')')).alias('p_value'),
    )
    # 居中对齐到 12 位
    result_df = result_df.with_columns(
        pl.col('mean').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('mean'),
        pl.col('t_stat').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('t_stat'),
        pl.col('p_value').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('p_value'),
    )

    # 合并为一行展示
    result_df = result_df.select(
        pl.col('variable'),
        (pl.col('mean') + pl.lit('\n') + pl.col('t_stat') + pl.lit('\n') + pl.col('p_value')).alias('stats'),
    )

    n_periods_df = pl.DataFrame(
        {
            'variable': ['观测数'],
            'stats': [n_periods]
        }
    ).with_columns(
        pl.col('stats').cast(pl.Utf8).alias('stats')
    )


    result_df = result_df.vstack(n_periods_df)


    return result_df

### 单AED变量

In [21]:
ma_result = fm_reg(ma_df, 'return', ['MA'])
with pl.Config(tbl_width_chars=200, tbl_cols=20, fmt_str_lengths=200):
    display(ma_result)

variable,stats
str,str
"""const""",""" -0.07401 [-9.805] (1.075e-22) """
"""MA""",""" 0.121 [11.21] (3.767e-29) """
"""观测数""","""241"""


### AED,BETA,SIZE,BM

In [22]:
ma_beta_size_bm_result = fm_reg(ma_df, 'return', ['MA','betavals','ln_market_value','bm_ratio'])
with pl.Config(tbl_width_chars=200, tbl_cols=20, fmt_str_lengths=200):
    display(ma_beta_size_bm_result)


variable,stats
str,str
"""const""",""" -0.1471 [-6.724] (1.772e-11) """
"""MA""",""" 0.1357 [13.92] (5.062e-44) """
"""betavals""",""" -0.003468 [-1.324] (0.1854) """
"""ln_market_value""",""" 0.005277 [4.581] (4.628e-06) """
"""bm_ratio""",""" -0.01906 [-4.787] (1.694e-06) """
"""观测数""","""241"""


### AED,BETA,SIZE,BM,GM,IA


In [23]:
ma_beta_size_bm_gm_ia_result = fm_reg(ma_df, 'return', ['MA','betavals','ln_market_value','bm_ratio','gross_margin','investment_ratio'])
with pl.Config(tbl_width_chars=200, tbl_cols=20, fmt_str_lengths=200):
    display(ma_beta_size_bm_gm_ia_result)

variable,stats
str,str
"""const""",""" -0.1439 [-6.086] (1.156e-09) """
"""MA""",""" 0.1352 [13.88] (7.914e-44) """
"""betavals""",""" -0.003595 [-1.384] (0.1665) """
"""ln_market_value""",""" 0.005293 [4.633] (3.6e-06) """
"""bm_ratio""",""" -0.02121 [-5.315] (1.069e-07) """
"""gross_margin""",""" -0.007526 [-3.081] (0.002063) """
"""investment_ratio""",""" 0.0008738 [0.1777] (0.859) """
"""观测数""","""241"""


拼在一起

In [24]:
fm_result = ma_beta_size_bm_gm_ia_result.join(ma_beta_size_bm_result,on='variable',how='left')
fm_result = fm_result.rename({'stats':'6控制变量','stats_right':'4控制变量'})
fm_result = fm_result.join(ma_result,on='variable',how='left')
fm_result = fm_result.rename({'stats':'单控制变量'})
fm_result = fm_result.select(
    pl.col('variable'),
    pl.col('单控制变量'),
    pl.col('4控制变量'),
    pl.col('6控制变量'),
)

# 移动const
fm_result = fm_result[[1,2,3,4,5,6,0,7],:]


In [25]:
with pl.Config(tbl_width_chars=200, tbl_cols=20, fmt_str_lengths=200):
    display(fm_result)

variable,单控制变量,4控制变量,6控制变量
str,str,str,str
"""MA""",""" 0.121 [11.21] (3.767e-29) """,""" 0.1357 [13.92] (5.062e-44) """,""" 0.1352 [13.88] (7.914e-44) """
"""betavals""",null,""" -0.003468 [-1.324] (0.1854) """,""" -0.003595 [-1.384] (0.1665) """
"""ln_market_value""",null,""" 0.005277 [4.581] (4.628e-06) """,""" 0.005293 [4.633] (3.6e-06) """
"""bm_ratio""",null,""" -0.01906 [-4.787] (1.694e-06) """,""" -0.02121 [-5.315] (1.069e-07) """
"""gross_margin""",null,null,""" -0.007526 [-3.081] (0.002063) """
"""investment_ratio""",null,null,""" 0.0008738 [0.1777] (0.859) """
"""const""",""" -0.07401 [-9.805] (1.075e-22) """,""" -0.1471 [-6.724] (1.772e-11) """,""" -0.1439 [-6.086] (1.156e-09) """
"""观测数""","""241""","""241""","""241"""


In [26]:
if SAVE:
    fm_result.write_parquet(SAVE_BASE_DIR + '/FM回归.parquet')